<a href="https://colab.research.google.com/github/sanima2025/sanima-fichas/blob/main/facturacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reporte de Facturación 2025

In [1]:
from google.colab import files
uploaded = files.upload()

Saving fresh-ember-461920-r3-fe9788193cba.json to fresh-ember-461920-r3-fe9788193cba.json


In [2]:
import gspread
from oauth2client.service_account import ServiceAccountCredentials

# Definir el scope
scope = ['https://spreadsheets.google.com/feeds','https://www.googleapis.com/auth/drive']

# Conectar con tus credenciales
creds = ServiceAccountCredentials.from_json_keyfile_name(
    'fresh-ember-461920-r3-fe9788193cba.json', scope)

# Autorizar cliente de Google Sheets
client = gspread.authorize(creds)

In [3]:
spreadsheet = client.open('Reporte de Facturación 2025')

In [4]:
hojas = spreadsheet.worksheets()

In [5]:
import pandas as pd
dfs = []

# Recorremos solo hojas "Monto a pagar"
for hoja in hojas:
    # Obtener los valores de la hoja
    data = hoja.get_all_values()
    df = pd.DataFrame(data)

    if df.empty:
        continue

    # Asignar la primera fila como encabezado
    df.columns = df.iloc[0]
    df = df[1:].copy()

    # *** SUGERENCIA DE CAMBIO: Asegurar que los nombres de columna sean únicos ***
    # Esto es crucial si la primera fila de alguna hoja tiene nombres duplicados.
    df = df.loc[:,~df.columns.duplicated()].copy()

    # *** FIX: Limpiar nombres de columna eliminando espacios al inicio y final ***
    # Esto ayuda a evitar KeyErrors por inconsistencias en los nombres de columna.
    df.columns = df.columns.str.strip()

    df = df.dropna(how="all")
    df["Nombre_Hoja"] = hoja.title
    dfs.append(df)

# Unir en un solo DataFrame
# Ahora que cada df en la lista 'dfs' tiene columnas únicas y limpias,
# la concatenación y selección posterior debería funcionar.
df_consolidado = pd.concat(dfs, ignore_index=True)

In [6]:

# --- Mover la extracción de "Mes" aquí, DESPUÉS de concatenar ---
# Esta columna 'Mes' SÍ se crea y parece estar en df_consolidado
# según el error.
df_consolidado["Mes"] = df_consolidado["Nombre_Hoja"].str.extract(r"Monto a pagar (\w+25)")


# --- Diagnóstico: Imprimir columnas para verificar los nombres ---
# Agrega esta impresión ANTES de intentar seleccionar columnas para df_hist
print("Columnas en df_consolidado JUSTO ANTES de crear df_hist:")
print(df_consolidado.columns.tolist()) # Imprimir como lista para mejor lectura
print("-" * 30)




Columnas en df_consolidado JUSTO ANTES de crear df_hist:
['Código del Depositante', 'ID', 'Deuda_inicial', 'Monto a Pagar', 'Monto pagado', 'Monto Pagado BCP', 'Nombre del Depositante', 'Deuda a hoy', 'Dias transcurridos', 'Metodo de pago', 'Deuda_final', 'Mora', 'Estado', 'Acuerdo pago', 'facturado_mensual', 'Deuda', 'Item_mensual', 'Avance mayo', 'Seguimiento', 'Documento', 'Mes', 'Nombre_Hoja', 'Mayo', 'Estado mayo', 'Tipo de Documento', 'Número', 'Contacto/Identification Number', 'Contacto/Name', 'Fecha de factura', 'Nombre en pantalla', 'Total', 'Líneas de factura/Producto', 'Total a pagar', 'Avance abril', '#REF!', '', 'Avance marzo', 'Avance febrero', 'Zona', 'Comunidad', 'Total a pagr', 'Avance enero', 'Deuda incial', 'Deuda final']
------------------------------


In [7]:

# %%
# Diccionario para convertir nombres tipo "enero", "febrero" en ENE25, FEB25...
mes_map = {
    "enero": "ENE25", "febrero": "FEB25", "marzo": "MAR25",
    "abril": "ABR25", "mayo": "MAY25"
}


In [8]:
df_consolidado["Mes"] = df_consolidado["Nombre_Hoja"].str.extract(r"Monto a pagar (\w+25)")

In [9]:
# %%
columnas_historial = [
    "ID",
    "Nombre del Depositante",
    "Código del Depositante",
    "Mes",
    "Deuda_inicial",
    "Dias transcurridos",
    "Metodo de pago",
    "Deuda_final",
    "Mora",
    "Estado",
    "Acuerdo pago",
    "Seguimiento",
    "Documento",
    # --- FIX: Ensure these match the actual column names in df_consolidado ---
    # Replace "Facturado mes" and "item_facturado" with the exact names
    # you see when you print df_consolidado.columns.tolist()
    # Based on the data, the actual names are likely something like:
    "facturado_mensual", # <-- **REPLACE THIS WITH THE ACTUAL COLUMN NAME FROM YOUR PRINT OUTPUT**
    "Item_mensual" # <-- **REPLACE THIS WITH THE ACTUAL COLUMN NAME FROM YOUR PRINT OUTPUT**
]

# Ensure all column names in columnas_historial are present in df_consolidado's columns
missing_cols = [col for col in columnas_historial if col not in df_consolidado.columns]
if missing_cols:
    print(f"Error: Las siguientes columnas no se encontraron en df_consolidado: {missing_cols}")
    # You might want to stop execution here or handle this case
else:
    df_hist = df_consolidado[columnas_historial].copy()

In [10]:
df_hist = df_hist.drop_duplicates(subset=["ID", "Mes"], keep="first")


In [11]:
df_hist.head()

,ID,Nombre del Depositante,Código del Depositante,Mes,Deuda_inicial,Dias transcurridos,Metodo de pago,Deuda_final,Mora,Estado,Acuerdo pago,Seguimiento,Documento,facturado_mensual,Item_mensual
0,81543745,HUARACCO URLLUB DANITZA MARELY,81543745,MAY25,39,16,Yape Servicios,0,Con mora,UN MES,,COBRANZAS - NICOL,B002-00035161,39,Servicio Estandar
1,80997131,Fernandez Briceño Diana Del pilar,80997131,MAY25,39,28,Yape Negocios Aliados,0,Sin mora,UN MES,,COBRANZAS - NICOL,B002-00034558,39,Servicio Estandar
2,80647633,Clemencia Maxima Alarcon Suarez,80647633,MAY25,20,1,Yape Servicios,0,Sin mora,UN MES,,COBRANZAS - NICOL,B002-00034127,20,Recolección quincenal
3,80647126,Lozano Espinoza Jenny Patricia,80647126,MAY25,88,99999,Pendiente,88,Sin mora,RIESGO,,COBRANZAS - NICOL,B002-00034460,39,Servicio Estandar
4,80605742,MELGAR RAMIREZ MARILU,80605742,MAY25,39,14,Agente,0,Sin mora,UN MES,,COBRANZAS - NICOL,B002-00035246,39,Servicio Estandar


In [12]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=df_hist)

https://docs.google.com/spreadsheets/d/1GHYRs2l8PoA70QRa1HruP2ht427AAn_pjUeiw-0xR0E/edit#gid=0


In [13]:
df_consolidado["Mes"].unique()

array(['MAY25', nan, 'ABR25', 'MAR25', 'FEB25', 'ENE25'], dtype=object)

In [14]:

# Pivotear por usuario y mes: genera columnas como ENE25_Estado, FEB25_Deuda_final, etc.
df_pivot = df_hist.pivot_table(
    index=["ID", "Nombre del Depositante", "Código del Depositante"],
    columns="Mes",
    values=[
        "Deuda_inicial",
        "Dias transcurridos",
        "Metodo de pago", # Changed back to "Metodo de pago" to match columna_historial
        "Deuda_final",
        "Mora",
        "Estado",
        "Acuerdo pago", # Changed back to "Acuerdo pago" to match columna_historial
        "Seguimiento",
        "Documento",
        # Ensure these next two match the names you found when printing columns
        "facturado_mensual", # Assuming this was the correct name
        "Item_mensual" # Assuming this was the correct name
    ],
    aggfunc="first"
)


# Convertir el MultiIndex de columnas a un índice simple con nombres combinados
# Handle cases where there might be only one level in the MultiIndex (if values had only one item)
if isinstance(df_pivot.columns, pd.MultiIndex):
    # Combine the levels of the MultiIndex into a single string, e.g., ('Deuda_inicial', 'ENE25') becomes 'Deuda_inicial_ENE25'
    # Use a consistent separator, like underscore
    df_pivot.columns = [f'{level1}_{level2}' for level1, level2 in df_pivot.columns]

from itertools import product

# Orden manual de meses deseado
orden_meses = ["ENE25", "FEB25", "MAR25", "ABR25", "MAY25"]

# The columns are now flat strings like 'Deuda_inicial_ENE25'.
# We need to extract the variable part (e.g., 'Deuda_inicial')
# and the month part (e.g., 'ENE25').
# The split should happen on the *last* underscore.

variables = list(set(col.rsplit('_', 1)[0] for col in df_pivot.columns if '_' in col))

# Create the desired order: Variable_Month
columnas_ordenadas = []
for var in sorted(variables):
    for mes in orden_meses:
        col_name = f"{var}_{mes}"
        if col_name in df_pivot.columns:
            columnas_ordenadas.append(col_name)

# Reindex df_pivot to enforce the desired order.
# This should only include the columns that were successfully created by the pivot.
df_pivot = df_pivot[columnas_ordenadas]


# Reiniciar índice para obtener DataFrame plano
df_historial_usuarios = df_pivot.reset_index()

# Ensure the index columns are at the beginning
index_cols = ["ID", "Nombre del Depositante", "Código delDepositante"]

# Ensure index_cols actually exist in the dataframe after reset_index
# (They should, but good practice to check)
existing_index_cols = [col for col in index_cols if col in df_historial_usuarios.columns]

# Create the final column order
final_cols_order = existing_index_cols + [col for col in df_historial_usuarios.columns if col not in existing_index_cols]

# Reindex the final dataframe
df_historial_usuarios = df_historial_usuarios[final_cols_order]


df_historial_usuarios = df_historial_usuarios.groupby("ID", as_index=False).first()

# Ver resultados
df_historial_usuarios.head()

,ID,Nombre del Depositante,Código del Depositante,Acuerdo pago_ENE25,Acuerdo pago_FEB25,Acuerdo pago_MAR25,Acuerdo pago_ABR25,Acuerdo pago_MAY25,Deuda_final_ENE25,Deuda_final_FEB25,...,Seguimiento_ENE25,Seguimiento_FEB25,Seguimiento_MAR25,Seguimiento_ABR25,Seguimiento_MAY25,facturado_mensual_ENE25,facturado_mensual_FEB25,facturado_mensual_MAR25,facturado_mensual_ABR25,facturado_mensual_MAY25
0,,,,None,None,None,None,,None,None,...,None,None,None,None,,None,None,None,None,
1,00370837,DIOS DIOSES JOSE BALTER,00370837,None,None,None,None,1ER,None,None,...,None,None,None,None,#N/A,None,None,None,None,20
2,00838106,Nicasio Grande,00838106,,,,,,0,0,...,COBRANZAS - WENDY,COBRANZAS - WENDY,COBRANZAS - WENDY,COBRANZAS - WENDY,COBRANZAS - WENDY,51,51,51,51,51
3,00910618,Sangama Paima Olegario,00910618,,,,,,0,0,...,COBRANZAS - NICOL,PILOTO - MADE,PILOTO - NICOL,PILOTO - NICOL,COBRANZAS - NICOL,39,39,39,39,39
4,00933745,Lopez Ramirez Darlith,00933745,,,,,,0,0,...,COBRANZAS - NICOL,PILOTO - MADE,PILOTO - NICOL,PILOTO - NICOL,COBRANZAS - NICOL,39,39,39,39,39


In [15]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=df_historial_usuarios)

https://docs.google.com/spreadsheets/d/1xwwoUanXIGRim8Dk3Z8b_Bsm8VL6debMvtj3VMk4mHw/edit#gid=0


In [16]:
# Función resumen por fila
def resumen_usuario(row):
    resumen = {}

    estados = row[cols_estado].fillna("").astype(str).str.strip().str.upper()
    moras = row[cols_mora].fillna("").astype(str).str.strip().str.lower()
    seguimientos = row[cols_seguimiento].dropna().astype(str).str.strip().unique()


In [17]:
# Reemplaza por un ID real (como el que viste en el ticket: "70983146")
usuario_id = "77318451"

# Filtrar por ese ID
ficha_usuario = df_historial_usuarios[df_historial_usuarios["ID"] == usuario_id]

# Mostrar verticalmente para que se lea como ficha
if ficha_usuario.empty:
    print("❌ Usuario no encontrado.")
else:
    display(ficha_usuario.T)


,2602
ID,77318451
Nombre del Depositante,GALVEZ UTIA SCARLETH MAGDIEL SAYURI
Código del Depositante,77318451
Acuerdo pago_ENE25,
Acuerdo pago_FEB25,
Acuerdo pago_MAR25,
Acuerdo pago_ABR25,AP
Acuerdo pago_MAY25,
Deuda_final_ENE25,0
Deuda_final_FEB25,0


In [18]:
def ver_ficha_usuario_formato(ID_usuario, df=df_historial_usuarios):
    usuario = df[df["ID"] == ID_usuario]

    if usuario.empty:
        print("❌ Usuario no encontrado.")
        return

    usuario = usuario.squeeze()

    nombre = usuario["Nombre del Depositante"]
    codigo = usuario["Código del Depositante"]

    meses = ["ENE25", "FEB25", "MAR25", "ABR25", "MAY25"]

    columnas = [
        "Deuda_inicial",
        "Deuda_final",
        "Documento",
        "Estado",
        "Mora",
        "Seguimiento",
        "Dias transcurridos",
        "Metodo de pago",
        "facturado_mensual",
        "Item_mensual"
    ]

    data = []
    for mes in meses:
        fila = {"Mes": mes}
        for col in columnas:
            key = f"{col}_{mes}"
            fila[col] = usuario.get(key, "")

        # Interpretar el contenido del campo Acuerdo_pago
        acuerdo_raw = str(usuario.get(f"Acuerdo pago_{mes}", "")).upper()

        if "AP" in acuerdo_raw:
            fila["Acuerdo de pago"] = "Solicitó Acuerdo de pago"
        elif "DES" in acuerdo_raw:
            fila["Acuerdo de pago"] = "Solicitó desinstalarse"
        elif "DPD" in acuerdo_raw:
            fila["Acuerdo de pago"] = "Cobranzas solicitó su desinstalación"
        elif "1ER" in acuerdo_raw:
            fila["Acuerdo de pago"] = "Primer mes en el servicio"
        elif "BOD" in acuerdo_raw:
            fila["Acuerdo de pago"] = "Es parte de negocios aliados"
        else:
            fila["Acuerdo de pago"] = "Sin observación"

        data.append(fila)

    ficha_df = pd.DataFrame(data)

    print(f"🧾 Ficha del usuario: {nombre}")
    print(f"🆔 ID: {ID_usuario} | Código del Depositante: {codigo}")
    display(ficha_df)



In [19]:
# Llama a la función con el ID del usuario
ver_ficha_usuario_formato("77556834")  # ← Cambia este ID por uno real si deseas ver otra ficha


🧾 Ficha del usuario: RODRIGUEZ TRIBIÑOS ANGIE NAYELI
🆔 ID: 77556834 | Código del Depositante: 77556834


,Mes,Deuda_inicial,Deuda_final,Documento,Estado,Mora,Seguimiento,Dias transcurridos,Metodo de pago,facturado_mensual,Item_mensual,Acuerdo de pago
0,ENE25,None,None,None,None,None,None,None,None,None,None,Sin observación
1,FEB25,None,None,None,None,None,None,None,None,None,None,Sin observación
2,MAR25,None,None,None,None,None,None,None,None,None,None,Sin observación
3,ABR25,20,20,B002-00033509,UN MES,Sin mora,#N/A,99999,Pendiente,20,Servicio Estándar con desc. (2 recolección),Primer mes en el servicio
4,MAY25,59,0,B002-00035219,RIESGO,Sin mora,COBRANZAS - NICOL,1,Yape Servicios,39,Servicio Estandar,Sin observación


In [20]:
ver_ficha_usuario_formato("77318451")

🧾 Ficha del usuario: GALVEZ UTIA SCARLETH MAGDIEL SAYURI
🆔 ID: 77318451 | Código del Depositante: 77318451


,Mes,Deuda_inicial,Deuda_final,Documento,Estado,Mora,Seguimiento,Dias transcurridos,Metodo de pago,facturado_mensual,Item_mensual,Acuerdo de pago
0,ENE25,39,0,B002-00028100,UN MES,Sin mora,COBRANZAS - WENDY,14,Yape Negocios Aliados,39,Servicio Estandar,Sin observación
1,FEB25,39,0,B002-00029254,UN MES,Sin mora,COBRANZAS - WENDY,15,Yape Negocios Aliados,39,Servicio Estandar,Sin observación
2,MAR25,39,39,B002-00031052,UN MES,Sin mora,COBRANZAS - WENDY,99999,Pendiente,39,Servicio Estandar,Sin observación
3,ABR25,78,0,B002-00033077,RIESGO,Sin mora,COBRANZAS - WENDY,22,Yape Negocios Aliados,39,Servicio Estandar,Solicitó Acuerdo de pago
4,MAY25,39,0,B002-00034737,UN MES,Sin mora,COBRANZAS - WENDY,27,Yape Negocios Aliados,39,Servicio Estandar,Sin observación


In [21]:
from IPython.display import display
import ipywidgets as widgets

# Asegúrate de que df_historial_usuarios está definido
# (Ejecuta las celdas anteriores para cargar y procesar los datos)

# Crear un widget de entrada de texto para el ID del usuario
text_input = widgets.Text(
    value='', # Valor inicial vacío
    placeholder='Ingresa el número de ID del usuario', # Texto de ayuda
    description='ID Usuario:', # Etiqueta para el campo
    disabled=False # Campo habilitado
)

# Crear un widget de botón
button = widgets.Button(
    description='Buscar Ficha', # Texto del botón
    disabled=False, # Botón habilitado
    button_style='', # Estilo del botón ('success', 'info', 'warning', 'danger' o '')
    tooltip='Haz clic para buscar la ficha del usuario', # Tooltip al pasar el mouse
    icon='search' # Icono del botón (opcional, requiere Font Awesome)
)

# Crear un widget de salida para mostrar el resultado
output_widget = widgets.Output()

# Función que se ejecutará cuando se haga clic en el botón
def on_button_clicked(b):
    with output_widget: # Redirigir la salida (print, display) a este widget
        output_widget.clear_output() # Limpiar la salida anterior
        user_id = text_input.value # Obtener el ID ingresado por el usuario
        if user_id: # Asegurarse de que se ingresó algo
            print(f"Buscando usuario con ID: {user_id}")
            ver_ficha_usuario_formato(user_id, df=df_historial_usuarios)
        else:
            print("Por favor, ingresa un número de ID.")

# Vincular la función al evento de clic del botón
button.on_click(on_button_clicked)

# Mostrar los widgets
print("Herramienta de Búsqueda de Fichas de Usuario")
display(text_input, button, output_widget)

Herramienta de Búsqueda de Fichas de Usuario


Text(value='', description='ID Usuario:', placeholder='Ingresa el número de ID del usuario')

Button(description='Buscar Ficha', icon='search', style=ButtonStyle(), tooltip='Haz clic para buscar la ficha …

Output()